# Querycraft v2 notebook A: 7B DoRA single pass on 2x T4 (fp16)

Settings: **GPU T4 x2**, **Internet on**. Inputs: `querycraft-code`, `querycraft-bird-eval`, `querycraft-7b-dora`.
Runs: (1) smoke test, (2) the 7B DoRA model single pass with the normal pruned schema, (3) single pass over the trained
retriever's tables. Both on dev and held back (nothing is chosen at 7B, so val is not run). These are the paired baselines for the 7B agent (notebook B),
measured on the same hardware and precision. Estimated 1.5 GPU hours. Output: `/kaggle/working/out/notebook_a.tar.gz`.


In [ ]:
import glob, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
def sh(cmd):
    print("$", cmd, flush=True); subprocess.run(cmd, shell=True, check=True)
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
IN = Path("/kaggle/input")
def find(slug):
    hits = [p for p in IN.glob("*") if slug in p.name] + [p for p in IN.glob("*/*/*") if slug in str(p)]
    assert hits, f"attach the dataset {slug} (Add Input)"; return hits[0]
W, T = Path("/kaggle/working"), Path("/kaggle/tmp")
QC = T / "qc"; OUT = W / "out"  # code + data in tmp (not saved); only out/ is kept
OUT.mkdir(exist_ok=True); T.mkdir(exist_ok=True)
def unpack(src, dst):
    """Kaggle may keep uploaded folders as .zip files: extract those, copy the rest."""
    dst.mkdir(parents=True, exist_ok=True)
    for p in Path(src).iterdir():
        if p.suffix == ".zip":
            zipfile.ZipFile(p).extractall(dst)
        elif p.is_dir():
            shutil.copytree(p, dst / p.name, dirs_exist_ok=True)
        else:
            shutil.copy(p, dst / p.name)
unpack(find("querycraft-code"), QC)
# Kaggle drops the top folder of a single uploaded zip: BIRD arrives as raw/ + processed/, the adapter as bare files
unpack(find("querycraft-bird-eval"), QC / "data")
if (QC / "data" / "data").is_dir():                 # in case a future upload keeps the folder
    shutil.copytree(QC / "data" / "data", QC / "data", dirs_exist_ok=True)
CKPT = T / "runs/qwen7b_dora_best/ckpt/checkpoint-560"
unpack(find("querycraft-7b-dora"), CKPT)
if (CKPT / "checkpoint-560").is_dir():
    shutil.copytree(CKPT / "checkpoint-560", CKPT, dirs_exist_ok=True)
assert (CKPT / "adapter_config.json").exists() and (QC / "data/processed/dev.jsonl").exists(), "unexpected dataset layout"
os.chdir(QC); os.environ["PYTHONPATH"] = str(QC); sys.path.insert(0, str(QC))
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
(QC / "runs").mkdir(exist_ok=True)
print(sorted(p.name for p in QC.iterdir()))


In [ ]:
# An isolated Python 3.12 venv with the laptop's exact stack (vLLM 0.32, torch 2.13 cu129). Kaggle's preinstalled
# packages (torchaudio, torchao, ...) are built for its own torch and break transformers/peft when mixed in.
VENV = T / "venv"; VPY = str(VENV / "bin/python")
sh("pip install -q uv")
sh(f"uv venv -q {VENV} --python 3.12")
sh(f"uv pip install --python {VPY} -q --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu129 "
   "'torch==2.13.0+cu129' 'vllm @ https://github.com/vllm-project/vllm/releases/download/v0.32.0/vllm-0.32.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl' "
   "'transformers>=5.16.1,<5.20.0' peft accelerate sqlglot pyyaml opentelemetry-api httpx")
sh(f"uv pip uninstall --python {VPY} torchcodec || true")


In [ ]:
# DoRA is not servable by vLLM: merge it into the 7B base on CPU (bf16 weights, ~15 GB in /kaggle/tmp)
t0 = time.time()
sh(f"{VPY} -c \"from querycraft.infer import merge_adapter; print(merge_adapter('Qwen/Qwen2.5-Coder-7B-Instruct', '{CKPT}'))\"")
MERGED = str(CKPT.parent.parent / "merged" / CKPT.name)
print(MERGED, f"{time.time() - t0:.0f} s")


In [ ]:
import yaml
# dev + held back only: nothing is chosen at 7B (the agent reuses the options tuned on val at 1.5B), and the weekly
# GPU quota may be 6 h, not 30 (Kaggle's quota API reported totalTimeAllowed 21600 s on 2026-11-23)
SPLITS = ["dev", "heldback"]
def cfg_7b(name, linking=None, extra=None):
    c = {"inherit": str(QC / "configs/rented/01_qwen7b_zeroshot.yaml"), "name": name, "model": MERGED,
         "eval": {"dtype": "float16", "tensor_parallel_size": 2, "gpu_memory_utilization": 0.85,
                  "tokenizer": "Qwen/Qwen2.5-Coder-7B-Instruct", "forgetting": False,
                  "splits": SPLITS}}
    if linking:
        c["prompt"] = {"linking": "retriever", "link_from": "runs/table_retriever/eval/{split}_tables.jsonl"}
    c.update(extra or {})
    p = QC / f"configs/kaggle_{name}.yaml"; p.write_text(yaml.safe_dump(c)); return str(p)


In [ ]:
# 5 minute smoke test: does vLLM 0.32 run on T4 (compute capability 7.5) in fp16 with tensor parallel 2?
# If this cell fails, stop here: the fallback is the Modal A100 after Dec 1 (docs/plans/v2_compute.md).
sh(f"{VPY} - <<'PY'\nimport sys; sys.path.insert(0, '{QC}')\nfrom querycraft.config import load_config\n"
   f"from querycraft.infer import Generator\nfrom querycraft.prompts import PromptBuilder\nfrom querycraft.data import load_split\n"
   f"c = load_config('{cfg_7b('smoke')}'); g = Generator(c, c['model']); pb = PromptBuilder(g.tok, 1280)\n"
   f"rows = load_split('dev')[:8]; import time; t = time.time(); out = g.generate([pb.prompt_text(r) for r in rows])\n"
   f"print(f'{{time.time() - t:.1f}} s for 8 prompts'); print(out[:3])\nPY")


In [ ]:
for name, link in (("qwen7b_dora_t4_single", False), ("qwen7b_dora_t4_retrieved", True)):
    t0 = time.time()
    sh(f"{VPY} -m querycraft.run_eval {cfg_7b(name, link)}")
    print(name, f"{(time.time() - t0) / 3600:.2f} h")


In [ ]:
def pack(name, paths):
    dst = OUT / f"{name}.tar.gz"
    sh(f"cd {QC} && tar czf {dst} " + " ".join(paths)); print(dst, dst.stat().st_size // 2**20, "MB")


In [ ]:
pack("notebook_a", ["runs/qwen7b_dora_t4_single", "runs/qwen7b_dora_t4_retrieved", "configs/kaggle_*.yaml"])
